**Silver Fact Visit Validation**

In [0]:
%sql
SELECT *
FROM healthcare_dev.silver.fact_visit;

In [0]:
# ============================================================
# GOLD LAYER — HOSPITAL DISEASE KPI
# Source: healthcare_dev.silver.fact_visit
# Target: healthcare_dev.gold.hospital_disease_kpi
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# 1. Read Silver fact table
df = spark.read.table("healthcare_dev.silver.fact_visit")

# 2. Define window by patient and admission date
w = (
    Window
    .partitionBy("patient_id")
    .orderBy("admission_date")
)

# 3. Calculate previous discharge and 30-day readmissions
df_with_prev = (
    df.withColumn(
        "previous_discharge",
        lag("discharge_date").over(w)
    )
    .withColumn(
        "days_since_last_visit",
        datediff("admission_date", "previous_discharge")
    )
    .withColumn(
        "is_readmission_30d",
        when(
            col("days_since_last_visit").between(0, 30),
            1
        ).otherwise(0)
    )
)

display(df_with_prev)

# 4. Aggregate hospital and diagnosis KPIs
gold_df = (
    df_with_prev
    .groupBy(
        "hospital_id",
        "hospital_name",
        "diagnosis_desc"
    )
    .agg(
        count("*").alias("total_visits"),
        sum("is_readmission_30d").alias("total_readmissions"),
        round(
            sum("is_readmission_30d") / count("*"), 3
        ).alias("readmission_rate"),
        sum("cost").alias("total_cost"),
        avg("cost").alias("avg_cost")
    )
    .withColumn(
        "gold_load_timestamp",
        current_timestamp()
    )
)

display(gold_df)

# 5. Write Gold Delta table
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("healthcare_dev.gold.hospital_disease_kpi")
)

print("Gold hospital_disease_kpi table created successfully.")

**Which hospital has the highest readmission rate?**

In [0]:
%sql
SELECT
    hospital_name,
    readmission_rate
FROM healthcare_dev.gold.hospital_disease_kpi
ORDER BY readmission_rate DESC;

**Which disease has the highest number of readmissions?**

In [0]:
%sql
SELECT
    diagnosis_desc,
    SUM(total_readmissions) AS readmission
FROM healthcare_dev.gold.hospital_disease_kpi
GROUP BY diagnosis_desc
ORDER BY readmission DESC;